In [ ]:
import torch
import pandas as pd
import lancedb

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer


In [ ]:
db = lancedb.connect("./lancedb")

table = db.open_table("qa")

df = table.to_pandas()[["question","answer"]]

df.head()

In [ ]:
def format_qa(example):
    
    text = f"""
### Question:
{example['question']}

### Answer:
{example['answer']}
"""
    
    return {"text": text}

In [ ]:
dataset = Dataset.from_pandas(df)

dataset = dataset.map(format_qa)

dataset = dataset.remove_columns(["question","answer"])

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "v_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./qa-mistral",
    
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    
    learning_rate=2e-4,
    num_train_epochs=3,
    
    logging_steps=10,
    
    save_strategy="epoch",
    
    fp16=True,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_args,
)

In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained("./qa-mistral-lora")

tokenizer.save_pretrained("./qa-mistral-lora")

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

prompt = """
### Question:
What is LanceDB?

### Answer:
"""

print(pipe(prompt, max_new_tokens=100)[0]["generated_text"])

In [ ]:
from peft import PeftModel

merged_model = model.merge_and_unload()

merged_model.save_pretrained("./qa-mistral-merged")

In [ ]:
results = table.search(question).limit(3).to_pandas()